The key elements we need to replicate from the paper and original code are: the IOI dataset with pIOI and pABC distributions, the logit difference metric (IO logit minus S logit), mean ablation using pABC activations, and activation patching at the head level. The circuit they found has 26 heads across 7 functional classes 
The objective is to run the same head-level patching loop and apply our three criteria to the result.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json
import random
from pathlib import Path
from itertools import product
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer

torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

RESULTS_DIR = Path("../results/level2IOI")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Using device: cpu


Load model

In [2]:
model = HookedTransformer.from_pretrained(
    "gpt2",
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
)
model.cfg.use_attn_result = True
model.eval()
model.to(device)

print(f"Layers: {model.cfg.n_layers}  Heads: {model.cfg.n_heads}  d_model: {model.cfg.d_model}")
print(f"use_attn_result: {model.cfg.use_attn_result}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cpu
Layers: 12  Heads: 12  d_model: 768
use_attn_result: True


Build the IOI dataset

Following the paper exactly: pIOI uses two names (IO and S), pABC uses three unrelated random names. The paper uses 15 templates in both ABBA and BABA order.

In [3]:
NAMES = [
    "Mary", "John", "Alice", "Bob", "Emma", "Frank", "Grace", "Henry",
    "Iris", "Jack", "Kate", "Liam", "Mia", "Noah", "Olivia", "Paul",
    "Rose", "Sam", "Tina", "Victor", "Wendy", "Anna", "Brian", "Clara",
    "Elena", "Felix", "Gina", "Harry", "Jake", "Luna", "Mark", "Nina",
    "Oscar", "Pam", "Ray", "Sara", "Tom", "Vera", "Will", "Zoe"
]

PLACES  = ["store", "school", "park", "office", "market", "gym",
            "beach", "hotel", "library", "theater", "hospital", "mall"]

OBJECTS = ["drink", "book", "ball", "bag", "gift", "key",
            "pen", "hat", "ring", "card", "toy", "cup"]

# All templates use [IO] and [S] explicitly — no ambiguity
# [S] is the subject (appears twice), [IO] is the indirect object (predicted)
# These are the paper's BABA + ABBA templates translated to unambiguous notation
TEMPLATES = [
    # BABA: IO appears first in the clause, S appears twice
    "Then, [IO] and [S] went to the [PLACE]. [S] gave a [OBJECT] to",
    "Then, [IO] and [S] had a lot of fun at the [PLACE]. [S] gave a [OBJECT] to",
    "Then, [IO] and [S] were working at the [PLACE]. [S] decided to give a [OBJECT] to",
    "Then, [IO] and [S] were thinking about going to the [PLACE]. [S] wanted to give a [OBJECT] to",
    "After [IO] and [S] went to the [PLACE], [S] gave a [OBJECT] to",
    "When [IO] and [S] got a [OBJECT] at the [PLACE], [S] decided to give it to",
    "While [IO] and [S] were working at the [PLACE], [S] gave a [OBJECT] to",
    "While [IO] and [S] were commuting to the [PLACE], [S] gave a [OBJECT] to",
    "After the lunch, [IO] and [S] went to the [PLACE]. [S] gave a [OBJECT] to",
    "Afterwards, [IO] and [S] went to the [PLACE]. [S] gave a [OBJECT] to",
    "The [PLACE] [IO] and [S] went to had a [OBJECT]. [S] gave it to",
    "Friends [IO] and [S] found a [OBJECT] at the [PLACE]. [S] gave it to",
    # ABBA: S appears first in the clause, S appears twice
    "Then, [S] and [IO] went to the [PLACE]. [S] gave a [OBJECT] to",
    "Then, [S] and [IO] had a lot of fun at the [PLACE]. [S] gave a [OBJECT] to",
    "After [S] and [IO] went to the [PLACE], [S] gave a [OBJECT] to",
    "When [S] and [IO] got a [OBJECT] at the [PLACE], [S] decided to give it to",
    "While [S] and [IO] were working at the [PLACE], [S] gave a [OBJECT] to",
    "After the lunch, [S] and [IO] went to the [PLACE]. [S] gave a [OBJECT] to",
    "Afterwards, [S] and [IO] went to the [PLACE]. [S] gave a [OBJECT] to",
    "The [PLACE] [S] and [IO] went to had a [OBJECT]. [S] gave it to",
]

def build_dataset(N, use_abc=False, seed=42):
    rng = random.Random(seed)
    dataset = []
    for i in range(N):
        template = rng.choice(TEMPLATES)
        place    = rng.choice(PLACES)
        obj      = rng.choice(OBJECTS)

        if use_abc:
            # pABC: three unrelated names, no duplication signal
            a, b, c = rng.sample(NAMES, 3)
            prompt = (template
                        .replace("[IO]", a)
                        .replace("[S]", b, 1)   # first [S] → b
                        .replace("[S]", c)       # second [S] → c
                        .replace("[PLACE]", place)
                        .replace("[OBJECT]", obj))
            io_name, s_name = a, b
        else:
            io_name, s_name = rng.sample(NAMES, 2)
            prompt = (template
                        .replace("[IO]", io_name)
                        .replace("[S]", s_name)
                        .replace("[PLACE]", place)
                        .replace("[OBJECT]", obj))

        dataset.append({"prompt": prompt, "io": io_name,"s": s_name, "template": template})
    return dataset

N = 200
ioi_data = build_dataset(N, use_abc=False, seed=42)
abc_data = build_dataset(N, use_abc=True,  seed=42)

# Sanity check
ex = ioi_data[0]
print(f"IOI: {ex['prompt']!r}")
print(f"     IO={ex['io']}  S={ex['s']}: model should predict '{ex['io']}'")
ex2 = ioi_data[1]
print(f"IOI: {ex2['prompt']!r}")
print(f"     IO={ex2['io']}  S={ex2['s']}: model should predict '{ex2['io']}'")
print(f"\nABC: {abc_data[0]['prompt']!r}")

IOI: 'Then, Sam and Paul were thinking about going to the store. Paul wanted to give a cup to'
     IO=Sam  S=Paul: model should predict 'Sam'
IOI: 'While Grace and Ray were commuting to the park, Ray gave a cup to'
     IO=Grace  S=Ray: model should predict 'Grace'

ABC: 'Then, Sam and Paul were thinking about going to the store. Olivia wanted to give a cup to'


Tokenise and verify single-token names

In [4]:
def tokenize_dataset(data, model):
    valid = []
    for ex in data:
        io_tok = model.to_tokens(" " + ex["io"], prepend_bos=False).squeeze()
        s_tok  = model.to_tokens(" " + ex["s"],  prepend_bos=False).squeeze()
        if io_tok.ndim == 0 and s_tok.ndim == 0:
            valid.append({**ex, "io_tok": io_tok.item(), "s_tok":  s_tok.item()})
    return valid

ioi_valid = tokenize_dataset(ioi_data, model)
abc_valid  = tokenize_dataset(abc_data,  model)
n_valid = min(len(ioi_valid), len(abc_valid))
ioi_valid = ioi_valid[:n_valid]
abc_valid = abc_valid[:n_valid]
print(f"Valid examples: {n_valid}")

def batch_tokens(valid_data, model, device):
    tokens_list = [model.to_tokens(ex["prompt"], prepend_bos=True).squeeze() for ex in valid_data]
    max_len = max(t.shape[0] for t in tokens_list)
    # Pad with BOS token (50256) so != check works cleanly with real token 0
    padded = torch.full((len(tokens_list), max_len), 50256, dtype=torch.long)
    for i, t in enumerate(tokens_list):
        padded[i, :t.shape[0]] = t
    return padded.to(device)

ioi_tokens = batch_tokens(ioi_valid, model, device)
abc_tokens  = batch_tokens(abc_valid,  model, device)

# Compute sequence lengths (pad token is 50256 = BOS, which we used as sentinel)
# Actually use a cleaner approach: store lengths directly
seq_lens = torch.tensor(
    [model.to_tokens(ex["prompt"], prepend_bos=True).shape[-1] for ex in ioi_valid], dtype=torch.long)
final_positions = (seq_lens - 1).to(device)   # index of last real token

# Verify one example
i = 0
last = final_positions[i].item()
last_token = model.to_single_str_token(ioi_tokens[i, last].item())
print(f"\nExample {i}:")
print(f"  Prompt: {ioi_valid[i]['prompt']!r}")
print(f"  Last token (pos {last}): {last_token!r}  (should be 'to')")
print(f"  IO token: {model.to_single_str_token(ioi_valid[i]['io_tok'])!r}")
print(f"  S  token: {model.to_single_str_token(ioi_valid[i]['s_tok'])!r}")

Valid examples: 200

Example 0:
  Prompt: 'Then, Sam and Paul were thinking about going to the store. Paul wanted to give a cup to'
  Last token (pos 20): ' to'  (should be 'to')
  IO token: ' Sam'
  S  token: ' Paul'


Logit difference metric

The paper uses logit difference (IO logit − S logit at the final token position) as the primary metric. This is cleaner than raw accuracy for measuring how strongly the model prefers the correct name.

In [7]:
def logit_diff(logits, valid_data, fps):
    """
    logits: [batch, seq, vocab]
    fps:    [batch] index of last real token per example
    """
    diffs = []
    for i, ex in enumerate(valid_data):
        io_logit = logits[i, fps[i].item(), ex["io_tok"]].item()
        s_logit  = logits[i, fps[i].item(), ex["s_tok"]].item()
        diffs.append(io_logit - s_logit)
    return torch.tensor(diffs)

BATCH_SIZE = 16

all_logits = []

with torch.no_grad():
    for start in range(0, len(ioi_tokens), BATCH_SIZE):
        end = start + BATCH_SIZE
        batch = ioi_tokens[start:end]

        logits = model(batch)
        all_logits.append(logits.cpu())

        # important for VRAM fragmentation
        del logits
        torch.cuda.empty_cache()

baseline_logits = torch.cat(all_logits, dim=0).to(device)
ld = logit_diff(baseline_logits, ioi_valid, final_positions)
print(f"Mean logit difference on pIOI: {ld.mean():.4f}  (paper reports ~3.56)")
print(f"IO predicted over S: {(ld > 0).float().mean():.1%}")

# Quick check, show top 5 predictions for first example
i = 0
top5 = baseline_logits[i, final_positions[i].item()].topk(5)
print(f"\nTop 5 predictions for example {i}:")
for val, idx in zip(top5.values, top5.indices):
    print(f"  {model.to_single_str_token(idx.item())!r:12}  {val.item():.2f}")
print(f"  (correct answer: {ioi_valid[i]['io']!r})")

Mean logit difference on pIOI: 3.9022  (paper reports ~3.56)
IO predicted over S: 99.0%

Top 5 predictions for example 0:
  ' Sam'        16.59
  ' the'        15.12
  ' them'       14.81
  ' a'          14.45
  ' his'        14.42
  (correct answer: 'Sam')


Compute pABC mean activations per head per template

The paper computes mean ablation using pABC activations averaged within each template to preserve grammatical structure. We implement this.

In [9]:
BATCH_SIZE = 8

N_LAYERS = model.cfg.n_layers
N_HEADS  = model.cfg.n_heads

RESULT_HOOK = lambda layer: f"blocks.{layer}.attn.hook_result"

print("Computing pABC mean activations...")

mean_abc_acts = {}

with torch.no_grad():

    for layer in tqdm(range(N_LAYERS)):

        hook_name = RESULT_HOOK(layer)

        running_sum = torch.zeros(
            N_HEADS,
            model.cfg.d_model,
            device=device
        )

        total_examples = 0

        for start in range(0, n_valid, BATCH_SIZE):

            end = min(start + BATCH_SIZE, n_valid)

            batch_tokens = abc_tokens[start:end]
            batch_fps    = final_positions[start:end]

            _, cache = model.run_with_cache(
                batch_tokens,
                names_filter=lambda name: name == hook_name
            )

            # [batch, seq, head, d_model]
            acts = cache[hook_name]

            # select final token position
            batch_indices = torch.arange(
                acts.shape[0],
                device=device
            )

            final_acts = acts[
                batch_indices,
                batch_fps
            ]  # [batch, head, d_model]

            running_sum += final_acts.sum(dim=0)
            total_examples += final_acts.shape[0]

            del cache
            del acts
            del final_acts

            torch.cuda.empty_cache()

        mean_abc_acts[layer] = running_sum / total_examples

print("Done.")
print(mean_abc_acts[0].shape)

Computing pABC mean activations...


  0%|          | 0/12 [00:00<?, ?it/s]

Done.
torch.Size([12, 768])


Clean and corrupted baselines

In [12]:
def make_full_ablation_hooks(batch_fps):

    hooks = []

    for layer in range(N_LAYERS):

        mean = mean_abc_acts[layer].to(device)

        def make_hook(m=mean, fp=batch_fps):

            def hook_fn(value, hook):

                # value: [batch, seq, head, d_model]

                batch_size = value.shape[0]

                value[
                    torch.arange(batch_size, device=value.device),
                    fp
                ] = m.unsqueeze(0).expand(batch_size, -1, -1)

                return value

            return hook_fn

        hooks.append(
            (RESULT_HOOK(layer), make_hook())
        )

    return hooks

In [13]:
BATCH_SIZE = 8

all_corrupted_logits = []

with torch.no_grad():

    for start in range(0, n_valid, BATCH_SIZE):

        end = min(start + BATCH_SIZE, n_valid)

        batch_tokens = ioi_tokens[start:end]
        batch_fps    = final_positions[start:end]

        corrupted_logits = model.run_with_hooks(
            batch_tokens,
            fwd_hooks=make_full_ablation_hooks(batch_fps)
        )

        all_corrupted_logits.append(
            corrupted_logits.cpu()
        )

        del corrupted_logits
        torch.cuda.empty_cache()

corrupted_logits = torch.cat(
    all_corrupted_logits,
    dim=0
).to(device)

: 

In [ ]:
clean_ld = logit_diff(
    baseline_logits,
    ioi_valid,
    final_positions
)

corrupted_ld = logit_diff(
    corrupted_logits,
    ioi_valid,
    final_positions
)

print(f"Clean     logit diff: {clean_ld.mean():.4f}")
print(f"Corrupted logit diff: {corrupted_ld.mean():.4f}")
print(f"Gap to recover:       {(clean_ld - corrupted_ld).mean():.4f}")

Expected behavior:

clean LD: +3 to +5
corrupted LD should collapse toward 0 or negative

If corrupted LD stays high, then the mean ablation is not actually overwriting the final-token head outputs correctly.

NOT YET EDITED

In [ ]:
def recovery_score(patched_ld, corrupted_ld, clean_ld):
    """Wang et al. 2022 Eq: fraction of clean-corrupted gap recovered."""
    gap = (clean_ld - corrupted_ld).mean().item()
    if abs(gap) < 1e-8:
        return 0.0
    return (patched_ld.mean().item() - corrupted_ld.mean().item()) / gap

# Clean: full model on pIOI
clean_ld = logit_diff(baseline_logits, ioi_valid, final_positions)

# Corrupted: mean-ablate ALL heads using pABC means
def make_full_ablation_hooks(final_positions):
    hooks = []
    for layer in range(N_LAYERS):
        mean = mean_abc_acts[layer].to(device)   # [n_heads, d_model]
        fps  = final_positions

        def make_hook(m=mean, fp=fps):
            def hook_fn(value, hook):
                # value: [batch, seq, n_heads, d_model]
                value[torch.arange(n_valid), fp] = m.unsqueeze(0).expand(n_valid, -1, -1)
                return value
            return hook_fn

        hooks.append((RESULT_HOOK(layer), make_hook()))
    return hooks

final_positions = torch.tensor(
    [(ioi_tokens[i] != 0).sum().item() - 1 for i in range(n_valid)],
    device=device)

with torch.no_grad():
    corrupted_logits = model.run_with_hooks(
        ioi_tokens,
        fwd_hooks=make_full_ablation_hooks(final_positions))
corrupted_ld = logit_diff(corrupted_logits, ioi_valid)

print(f"Clean     logit diff: {clean_ld.mean():.4f}")
print(f"Corrupted logit diff: {corrupted_ld.mean():.4f}")
print(f"Gap to recover:       {(clean_ld - corrupted_ld).mean():.4f}")